# Intelligent Agents: Reflex-Based Agents for GridHunt (Multi-agent Environment)

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, n = 30, max_steps = 100, visualize = False, animation = False):
    """
    Simulate an arena where two hunter agent tries to catch a monster agent.

    Parameters:
    hunter_1_agent_function and hunter_2_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('east')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[2 3]'; Hunter 2 is at '[1 1]'; Monster is at '[1 2]'

.....
.2M..
...1.
.....
.....
Hunter 1 chose action 'teleport'
Hunter 2 chose action 'east'
Monster chose action 'east'

=====

Step 1:
Hunter 1 is at '[1 2]'; Hunter 2 is at '[1 2]'; Monster is at '[1 3]'

.....
..2M.
.....
.....
.....
Hunter 1 chose action 'north'
Hunter 2 chose action 'teleport'
Monster chose action 'south'

=====

Step 2:
Hunter 1 is at '[0 2]'; Hunter 2 is at '[2 3]'; Monster is at '[2 3]'

..1..
.....
...2.
.....
.....
Hunter 2 caught the monster in 2 steps!


('2', 2)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [7]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 34), ('2', 38), ('1', 29), ('M', nan), ('M', nan), ('M', nan), ('2', 42), ('1', 82), ('M', nan), ('1', 96), ('M', nan), ('2', 24), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 19), ('1', 74), ('M', nan), ('1', 55), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 75), ('M', nan), ('M', nan), ('2', 87), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 53), ('M', nan), ('1', 56), ('M', nan), ('M', nan), ('M', nan), ('2', 10), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 30), ('M', nan), ('M', nan), ('2', 93), ('M', nan), ('M', nan), ('1', 46), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 53), ('1', 17), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 42), ('M', nan), ('1', 92), ('M', nan), ('1', 58), ('1', 13), ('M', nan), ('2', 2), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 17), ('M', nan), ('M', nan), ('M', nan), ('1', 

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [8]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 17
Hunter 2 won: 15
Monster 1 won: 68


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [9]:
def hunter_agent_function(hunter_location, monster_location):
    if monster_location is not None:
        if hunter_location[0] < monster_location[0]:
            return 'south'
        if hunter_location[0] > monster_location[0]:
            return 'north'
        if hunter_location[1] < monster_location[1]:
            return 'east'
        if hunter_location[1] > monster_location[1]:
            return 'west'
        else:
            return 'teleport'

In [10]:
def hunter_teleport_function(hunter_location, monster_location):
    distance = (
        abs(monster_location[0] - hunter_location[0])
        + abs(monster_location[1] - hunter_location[1])
    )

    if distance > 15:
        return 'teleport'

    if monster_location is not None:
        if hunter_location[0] < monster_location[0]:
            return 'south'
        if hunter_location[0] > monster_location[0]:
            return 'north'
        if hunter_location[1] < monster_location[1]:
            return 'east'
        if hunter_location[1] > monster_location[1]:
            return 'west'
        else:
            return 'teleport'

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [11]:
hunter_1 = hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[0 2]'; Hunter 2 is at '[1 0]'; Monster is at '[3 2]'

..1..
2....
.....
..M..
.....
Hunter 1 chose action 'south'
Hunter 2 chose action 'teleport'
Monster chose action 'stay'

=====

Step 1:
Hunter 1 is at '[1 2]'; Hunter 2 is at '[1 3]'; Monster is at '[3 2]'

.....
..12.
.....
..M..
.....
Hunter 1 chose action 'south'
Hunter 2 chose action 'south'
Monster chose action 'east'

=====

Step 2:
Hunter 1 is at '[2 2]'; Hunter 2 is at '[2 3]'; Monster is at '[3 3]'

.....
.....
..12.
...M.
.....
Hunter 1 chose action 'south'
Hunter 2 chose action 'south'
Monster chose action 'stay'

=====

Step 3:
Hunter 1 is at '[3 2]'; Hunter 2 is at '[3 3]'; Monster is at '[3 3]'

.....
.....
.....
..12.
.....
Hunter 2 caught the monster in 3 steps!


('2', 3)

In [12]:
# Experiment code (20x20 grid, 100 runs, max 100 steps): greedy hunter vs randomized hunter
hunter_1 = hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2,    
                           monster_agent_function_simple, 
                           n=20, max_steps=40, visualize=False) for _ in range(100)] 
print(steps)

[('1', 12), ('1', 11), ('1', 7), ('1', 27), ('1', 11), ('1', 11), ('1', 15), ('1', 19), ('1', 12), ('1', 17), ('1', 14), ('1', 20), ('1', 33), ('1', 13), ('1', 13), ('1', 18), ('1', 5), ('1', 16), ('1', 16), ('1', 25), ('1', 13), ('1', 12), ('1', 37), ('1', 26), ('1', 25), ('1', 9), ('1', 7), ('M', nan), ('1', 21), ('1', 11), ('1', 26), ('1', 28), ('1', 23), ('1', 17), ('1', 31), ('1', 35), ('1', 24), ('1', 11), ('1', 7), ('2', 24), ('1', 4), ('1', 20), ('1', 4), ('1', 25), ('1', 10), ('1', 25), ('M', nan), ('1', 21), ('1', 8), ('M', nan), ('1', 10), ('1', 12), ('1', 15), ('1', 12), ('1', 18), ('1', 13), ('1', 34), ('1', 8), ('1', 8), ('1', 9), ('1', 21), ('1', 10), ('1', 8), ('1', 4), ('2', 9), ('1', 35), ('1', 14), ('1', 4), ('1', 2), ('1', 16), ('1', 32), ('1', 18), ('1', 20), ('1', 26), ('1', 6), ('M', nan), ('1', 34), ('1', 7), ('1', 18), ('1', 13), ('1', 11), ('1', 11), ('1', 2), ('1', 31), ('1', 31), ('1', 25), ('1', 16), ('1', 24), ('1', 11), ('1', 6), ('1', 18), ('1', 16), ('1

In [13]:
# Summary of results
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       

Hunter 1 won: 94
Hunter 2 won: 2
Monster 1 won: 4


In [14]:
# Experiment code (50x50 grid, 100 runs, max 100 steps): greedy hunter vs randomized hunter
hunter_1 = hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=50, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

# Summary of results
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       

[('1', 6), ('1', 70), ('1', 26), ('1', 32), ('1', 35), ('1', 36), ('1', 24), ('1', 76), ('1', 31), ('1', 87), ('1', 20), ('1', 54), ('1', 10), ('1', 17), ('1', 25), ('1', 46), ('1', 39), ('M', nan), ('1', 28), ('1', 26), ('1', 29), ('1', 17), ('1', 42), ('1', 10), ('1', 61), ('1', 32), ('1', 17), ('1', 7), ('1', 35), ('1', 12), ('1', 46), ('1', 52), ('1', 42), ('1', 40), ('1', 82), ('1', 49), ('1', 81), ('1', 38), ('1', 23), ('1', 25), ('1', 37), ('1', 26), ('1', 9), ('1', 4), ('1', 67), ('1', 79), ('1', 30), ('1', 70), ('1', 59), ('1', 75), ('1', 35), ('1', 56), ('1', 6), ('M', nan), ('1', 20), ('1', 41), ('1', 4), ('1', 39), ('1', 54), ('1', 23), ('1', 33), ('1', 70), ('1', 23), ('1', 77), ('1', 56), ('1', 12), ('1', 11), ('1', 34), ('1', 56), ('1', 32), ('1', 6), ('1', 53), ('M', nan), ('1', 42), ('1', 29), ('1', 21), ('1', 4), ('1', 66), ('1', 20), ('1', 19), ('M', nan), ('1', 11), ('1', 41), ('1', 25), ('1', 38), ('1', 59), ('1', 23), ('1', 16), ('1', 33), ('1', 17), ('M', nan), (

In [15]:
# Experiment code (20x20 grid, 100 runs, max 40 steps): greedy hunter vs teleport hunter
hunter_1 = hunter_agent_function
hunter_2 = hunter_teleport_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=40, visualize=False) for _ in range(100)]
print(steps)

# Summary of results
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")

[('2', 8), ('2', 13), ('2', 3), ('2', 19), ('1', 14), ('1', 7), ('1', 6), ('1', 10), ('1', 18), ('2', 12), ('2', 18), ('2', 6), ('1', 13), ('1', 10), ('1', 14), ('1', 5), ('1', 6), ('1', 3), ('2', 14), ('1', 10), ('1', 9), ('2', 5), ('1', 13), ('1', 15), ('2', 6), ('1', 10), ('2', 20), ('1', 1), ('1', 10), ('2', 11), ('1', 9), ('1', 16), ('2', 15), ('1', 6), ('2', 14), ('2', 6), ('2', 11), ('2', 15), ('1', 13), ('1', 3), ('2', 13), ('1', 8), ('2', 6), ('2', 2), ('1', 4), ('1', 18), ('1', 9), ('1', 10), ('2', 13), ('2', 3), ('2', 9), ('2', 11), ('2', 13), ('1', 27), ('1', 12), ('1', 30), ('1', 10), ('2', 19), ('2', 14), ('2', 4), ('2', 5), ('2', 11), ('1', 11), ('1', 12), ('2', 4), ('1', 12), ('2', 18), ('2', 18), ('2', 37), ('2', 4), ('1', 16), ('2', 7), ('2', 9), ('2', 18), ('1', 11), ('2', 5), ('2', 10), ('2', 5), ('2', 13), ('2', 9), ('1', 15), ('2', 11), ('2', 16), ('2', 14), ('1', 17), ('2', 7), ('1', 15), ('2', 3), ('2', 5), ('2', 17), ('1', 15), ('2', 9), ('2', 2), ('1', 5), ('1

In [16]:
# Experiment code (50x50 grid, 100 runs, max 100 steps): greedy hunter vs teleport hunter
hunter_1 = hunter_agent_function
hunter_2 = hunter_teleport_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=50, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

# Summary of results
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")

[('2', 19), ('2', 31), ('2', 12), ('2', 25), ('1', 21), ('2', 3), ('2', 29), ('2', 17), ('1', 12), ('2', 20), ('1', 4), ('2', 16), ('2', 14), ('2', 22), ('2', 26), ('2', 11), ('2', 10), ('2', 18), ('2', 21), ('2', 17), ('2', 19), ('2', 17), ('2', 9), ('2', 29), ('1', 14), ('1', 20), ('2', 16), ('2', 17), ('2', 8), ('2', 28), ('2', 3), ('2', 22), ('2', 11), ('2', 12), ('1', 7), ('2', 8), ('1', 9), ('2', 15), ('1', 1), ('2', 14), ('1', 28), ('1', 9), ('1', 14), ('2', 21), ('2', 33), ('2', 17), ('2', 26), ('2', 40), ('2', 13), ('2', 31), ('2', 20), ('2', 14), ('2', 14), ('2', 7), ('2', 12), ('1', 5), ('2', 31), ('2', 8), ('2', 56), ('2', 19), ('1', 13), ('2', 16), ('1', 10), ('2', 2), ('2', 11), ('2', 44), ('1', 29), ('2', 28), ('2', 20), ('2', 38), ('2', 14), ('2', 22), ('2', 11), ('2', 4), ('2', 9), ('1', 26), ('2', 6), ('2', 37), ('2', 12), ('1', 23), ('2', 12), ('2', 51), ('2', 24), ('2', 3), ('2', 15), ('2', 27), ('2', 17), ('1', 29), ('2', 11), ('2', 23), ('2', 20), ('2', 23), ('2',

## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

### Discussion
1. What is your hunter's final strategy to choose actions. Why does it work well?
The hunters use two different strategies based on the monster's location.
The first hunter always moves directly toward the monster by choosing the direction that reduces the Manhattan distance. The second hunter uses teleportation when the distance to the monster is large, then switches to direct movement when it gets closer.
These strategies work well because the hunters use information about the monster position instead of moving randomly. The direct movement strategy provides a reliable way to catch the monster, while teleportation can reduce the searching time when the initial distance is large.

2. Do you use teleportation. Why and in what situation? Why not?
One hunter uses teleportation when the distance to the monster is greater than 15 cells. This helps reduce the time needed to approach the monster when they start far apart.
However, teleportation is not always effective because it moves the hunter to a random position. If used when the hunter is already close to the monster, it may increase the distance and waste progress. Therefore, normal movement is preferred during the final chase.

3. How does the arena size affect your hunter agent's performance?
The arena size affects the number of steps required to catch the monster. In smaller arenas, hunters can reach the monster faster because the initial distance is limited.
In larger arenas, the monster has more space to escape and hunters need more steps to catch it. The teleport-based strategy can become more useful in larger environments because it helps reduce the effect of large initial distances.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

### Discussion
The monster agent could be improved by using the positions of both hunters to make better decisions. Instead of moving randomly, the monster can detect which hunter is closer and move away from that hunter to increase the distance. It can also avoid staying still and use unpredictable movements when the hunters are far away. This strategy would make the monster harder to catch and increase its survival time.

# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.